In [ ]:
import pandas as pd


uci_path = r"C:\Project\data_raw\SMSSpamCollection"


df = pd.read_csv(uci_path, sep="\t", header=None, names=["label", "text"])


df = df.dropna(subset=["text"])
df = df.drop_duplicates(subset=["text"])
df["label"] = df["label"].str.lower().str.strip()
df = df[df["label"].isin(["ham", "spam"])]
df["label"] = df["label"].map({"ham": 0, "spam": 1}).astype("int8")

print(df.head())
print("📌 Sample Number:", len(df))
print("📌 label ratio:\n", df["label"].value_counts(normalize=True))


df.to_csv(r"C:\Project\data_work\step0_sms_loaded.csv", index=False)


   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
📌 Number: 5169
📌 label ratio:
 label
0    0.87367
1    0.12633
Name: proportion, dtype: float64


In [ ]:
import pandas as pd


uci_path = r"C:\Project\data_raw\SMSSpamCollection"


df = pd.read_csv(uci_path, sep="\t", header=None, names=["label","text"])


df = df.dropna(subset=["text"])
df = df.drop_duplicates(subset=["text"])
df["label"] = df["label"].str.lower().str.strip()
df = df[df["label"].isin(["ham","spam"])]
df["label"] = df["label"].map({"ham":0, "spam":1}).astype("int8")

print(df.head())
print("📊 label ratio:", df["label"].value_counts(normalize=True))


df.to_csv(r"C:\Project\data_work\step0_sms_loaded.csv", index=False)



   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...
📊 Tỷ lệ nhãn: label
0    0.87367
1    0.12633
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
import re
import os


src_path = r"C:\Project\data_work\step0_sms_loaded.csv"
dst_path = r"C:\Project\data_work\step1_text_clean.csv"
os.makedirs(r"C:\Project\data_work", exist_ok=True)

df = pd.read_csv(src_path)

def basic_clean(s: str) -> str:
    s = s.lower() 
    s = re.sub(r"http\S+|www\.\S+", " [url] ", s) 
    s = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", " [email] ", s) 
    s = re.sub(r"\d+", " [num] ", s)  
    s = re.sub(r"[^a-z\s\[\]]", " ", s)  
    s = re.sub(r"\s+", " ", s).strip() 
    return s


df["text_clean"] = df["text"].astype(str).apply(basic_clean)


df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)


print(df[["label","text","text_clean"]].head(8))


df.to_csv(dst_path, index=False)
print("✅ Saved:", dst_path)


   label                                               text  \
0      0  Go until jurong point, crazy.. Available only ...   
1      0                      Ok lar... Joking wif u oni...   
2      1  Free entry in 2 a wkly comp to win FA Cup fina...   
3      0  U dun say so early hor... U c already then say...   
4      0  Nah I don't think he goes to usf, he lives aro...   
5      1  FreeMsg Hey there darling it's been 3 week's n...   
6      0  Even my brother is not like to speak with me. ...   
7      0  As per your request 'Melle Melle (Oru Minnamin...   

                                          text_clean  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in [num] a wkly comp to win fa cup ...  
3        u dun say so early hor u c already then say  
4  nah i don t think he goes to usf he lives arou...  
5  freemsg hey there darling it s been [num] week...  
6  even my brother is not like to speak with me

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os


src_path = r"C:\Project\data_work\step1_text_clean.csv"
final_dir = r"C:\Project\data_final"
os.makedirs(final_dir, exist_ok=True)

df = pd.read_csv(src_path)


X = df[["text", "text_clean"]]
y = df["label"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


train = X_train.copy()
train["label"] = y_train.values

test = X_test.copy()
test["label"] = y_test.values


train_path = os.path.join(final_dir, "train.csv")
test_path = os.path.join(final_dir, "test.csv")

train.to_csv(train_path, index=False)
test.to_csv(test_path, index=False)

print("✅ Train:", train.shape, "→", train_path)
print("✅ Test :", test.shape, "→", test_path)


print("\n📊 Train label ratio:")
print(train["label"].value_counts(normalize=True))
print("\n📊 Test label ratio:")
print(test["label"].value_counts(normalize=True))


✅ Train: (4133, 3) → C:\Project\data_final\train.csv
✅ Test : (1034, 3) → C:\Project\data_final\test.csv

📊 Train label ratio:
label
0    0.873699
1    0.126301
Name: proportion, dtype: float64

📊 Test label ratio:
label
0    0.873308
1    0.126692
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os


raw_path   = r"C:\Project\data_raw\emails.csv"
work_dir   = r"C:\Project\data_work"
final_dir  = r"C:\Project\data_final"

os.makedirs(work_dir, exist_ok=True)
os.makedirs(final_dir, exist_ok=True)


df = pd.read_csv(raw_path)
print("1) Loaded:", df.shape)


assert "Prediction" in df.columns, "Cannot find 'Prediction' in emails.csv"


print("\n2) Overview:")
print(" - Number of columns:", df.shape[1])
print(" - Label distribution (initial):")
print(df["Prediction"].value_counts(dropna=False))


na_rows = df.isna().any(axis=1).sum()
na_cells = df.isna().sum().sum()
print(f"\n3) NA check: {na_rows} rows have NA, {na_cells} cells have NA")


feat_cols = [c for c in df.columns if c != "Prediction"]
if na_cells > 0:
    df[feat_cols] = df[feat_cols].fillna(0)
    if df["Prediction"].isna().any():
        before = len(df)
        df = df.dropna(subset=["Prediction"])
    print(f" - remove {before - len(df)} rows has NA in label")
    print(" - did fillna(0) for feature columns")


before = len(df)

df = df.drop_duplicates()
print(f"\n4) Duplicates: remove {before - len(df)} duplicate rows (if any)")


df["Prediction"] = df["Prediction"].astype("int8")


X = df.drop(columns=["Prediction"])
y = df["Prediction"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

train_email = X_train.copy()
train_email["Prediction"] = y_train.values

test_email  = X_test.copy()
test_email["Prediction"]  = y_test.values


train_path = os.path.join(final_dir, "emails_train.csv")
test_path  = os.path.join(final_dir, "emails_test.csv")

train_email.to_csv(train_path, index=False)
test_email.to_csv(test_path, index=False)


df.to_csv(os.path.join(work_dir, "emails_step_cleaned.csv"), index=False)


print("\n5) result:")
print(" - Train:", train_email.shape, "→", train_path)
print(" - Test :", test_email.shape,  "→", test_path)
print("\n - Train label distribution:")
print(train_email["Prediction"].value_counts(normalize=True))
print("\n - Test label distribution:")
print(test_email["Prediction"].value_counts(normalize=True))


1) Loaded: (5172, 3002)

2) Tổng quan:
 - Number of columns: 3002
 - Label distribution (initial):
Prediction
0    3672
1    1500
Name: count, dtype: int64

3) NA check: 0 rows have NA, 0 cells have NA

4) Duplicates: remove 0 duplicate rows (if any)

5) result:
 - Train: (4137, 3002) → C:\Project\data_final\emails_train.csv
 - Test : (1035, 3002) → C:\Project\data_final\emails_test.csv

 - Train label distribution:
Prediction
0    0.709935
1    0.290065
Name: proportion, dtype: float64

 - Test label distribution:
Prediction
0    0.710145
1    0.289855
Name: proportion, dtype: float64
